In [1]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

c:\Users\knguy139\Documents\Projects\.venv\Lib\site-packages\snowflake\connector\vendored\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [3]:
df_2025 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
        and d.collection is not null
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

df_2026 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
        , count(distinct case when a.member_appeal_ovtn_ind = 1 then a.case_id end) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202601' and '202604'
        and d.collection is not null
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"df_2025: {df_2025.shape}")
print(f"df_2026: {df_2026.shape}")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVLRbtowFP2VyHtO4gTCiAVUFNYNFdoI0iH1ZXJiJ1g4drCdpvz9nABS99BKe7Bk2efcc%2B49d3L3XnHnjSrNpJiCwIPAoSKXhIlyCl7SB3cMHG2wIJhLQafgTDW4m000rniN5o05iC09NVQbxxYSGnUfU9AogSTWTCOBK6qRydFuvlmj0IMIa02VsXLgSiGaWa2DMTXy%2FbZtvXbgSVX6IYTQh7FvUR3kG%2FggUX%2BtUStpZC75jfJue%2FpEIvDhsJOwCKuQXIn3TFxG8JVKdgFp9CtNEzd53qXAmd%2B6W0ihm4qqHVVvLKcv2%2FXFgLYOmkPp2kNajPWfA%2Fa0kG3B8ZHmsqobY2t69uYXlPhclsxOarWcgvrISELw6dgmUXoK5Hq5Ln9uFuf7vaGvOt4Osx9PWfu4b7PN4%2F65zIHz%2B5Zr2OW60rqhK9GlaewTDEcuHLlBnAYRggMUDbwxjF%2BBs7RpMoFNz7xZ7n14FcuV1LIwUnAmaO%2BSZDAqcI7dfBxid5jFxM3iPHJhMRpmo%2B9RNAwDv8ssBJe9Qb0RNfu%2FaUz8j9zrAj7ZTFbLRHKWn50HqSpsPo8s8IL%2BhRG36KGIVpjxOSGKam2j41y2C0WxsXtuVEOBP7uo%2Frvps78%3D&RelayState=ver%3A3-hint%3A260092117886646-ETMsDgAAAZ7gaO7QABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEL74Z4qewLHAGChfr%2BVQ6QoAAACQYyyXamaZcsyu

In [4]:
def mom_optimal_k(successes, trials):
    mask = (trials > 0) & (successes >= 0)
    s, n = successes[mask].values, trials[mask].values
    p = s / n
    p_bar = p.mean()
    v_obs = p.var(ddof=1)
    n_bar = len(n) / np.sum(1.0 / n)
    v_sampling = p_bar * (1 - p_bar) / n_bar
    v_signal = v_obs - v_sampling
    if v_signal <= 0:
        return 50.0
    alpha_hat = p_bar * (p_bar * (1 - p_bar) / v_signal - 1)
    beta_hat = (1 - p_bar) * (p_bar * (1 - p_bar) / v_signal - 1)
    if alpha_hat <= 0 or beta_hat <= 0:
        return 50.0
    return alpha_hat + beta_hat


def compute_shrinkage(df, year, k_map, features_8=False):
    """Compute rates + shrinkage. If features_8=True, excludes member_appeal."""
    has_ma = ("member_appeal_cnt" in df.columns) and not features_8

    df = df.assign(
        initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count,
        persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count,
        p2p_overturn_rate = lambda x: x.p2p_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
        appeal_overturn_rate = lambda x: x.appeal_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
        mcr_overturn_rate = lambda x: x.mcr_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
        md_review_rate = lambda x: x.md_reviewed_cnt / x.case_count,
    )
    if has_ma:
        df = df.assign(
            member_appeal_rate = lambda x: x.member_appeal_cnt / x.initial_adr_cnt.replace(0, np.nan),
        )

    rate_cols = ["initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
                 "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"]
    if has_ma:
        rate_cols.append("member_appeal_rate")

    for col in rate_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

    df_broad = df.query("case_count >= 30").copy()
    gr_initial_adr = df_broad["initial_adr_cnt"].sum() / df_broad["case_count"].sum()
    gr_persistent = df_broad["persistent_adr_cnt"].sum() / df_broad["case_count"].sum()
    denom_adr_broad = df_broad.loc[df_broad["initial_adr_cnt"] > 0, "initial_adr_cnt"].sum()
    gr_p2p = df_broad["p2p_ovrtn_cnt"].sum() / denom_adr_broad if denom_adr_broad > 0 else 0
    gr_appeal = df_broad["appeal_ovrtn_cnt"].sum() / denom_adr_broad if denom_adr_broad > 0 else 0
    gr_mcr = df_broad["mcr_ovrtn_cnt"].sum() / denom_adr_broad if denom_adr_broad > 0 else 0
    gr_md = df_broad["md_reviewed_cnt"].sum() / df_broad["case_count"].sum()

    df_iso = df.query("case_count >= 30 and initial_adr_cnt >= 10").copy()

    df_iso["initial_adr_adj_rate"] = (df_iso["initial_adr_cnt"] + k_map["initial_adr"] * gr_initial_adr) / (df_iso["case_count"] + k_map["initial_adr"])
    df_iso["persistent_adr_adj_rate"] = (df_iso["persistent_adr_cnt"] + k_map["persistent_adr"] * gr_persistent) / (df_iso["case_count"] + k_map["persistent_adr"])
    df_iso["p2p_overturn_adj_rate"] = (df_iso["p2p_ovrtn_cnt"] + k_map["p2p_overturn"] * gr_p2p) / (df_iso["initial_adr_cnt"] + k_map["p2p_overturn"])
    df_iso["appeal_overturn_adj_rate"] = (df_iso["appeal_ovrtn_cnt"] + k_map["appeal_overturn"] * gr_appeal) / (df_iso["initial_adr_cnt"] + k_map["appeal_overturn"])
    df_iso["mcr_overturn_adj_rate"] = (df_iso["mcr_ovrtn_cnt"] + k_map["mcr_overturn"] * gr_mcr) / (df_iso["initial_adr_cnt"] + k_map["mcr_overturn"])
    df_iso["md_review_adj_rate"] = (df_iso["md_reviewed_cnt"] + k_map["md_review"] * gr_md) / (df_iso["case_count"] + k_map["md_review"])
    df_iso["log_case_count"] = np.log1p(df_iso["case_count"])
    df_iso["log_initial_adr_cnt"] = np.log1p(df_iso["initial_adr_cnt"])

    if has_ma:
        gr_ma = df_broad["member_appeal_cnt"].sum() / denom_adr_broad if denom_adr_broad > 0 else 0
        df_iso["member_appeal_adj_rate"] = (df_iso["member_appeal_cnt"] + k_map["member_appeal"] * gr_ma) / (df_iso["initial_adr_cnt"] + k_map["member_appeal"])

    df_iso["year"] = year
    return df_iso

In [5]:
df_all = pd.concat([
    df_2025.query("case_count >= 30").assign(
        initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count
    ).query("initial_adr_cnt >= 10"),
    df_2026.query("case_count >= 30").assign(
        initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count
    ).query("initial_adr_cnt >= 10")
], ignore_index=True)

rate_configs = {
    "initial_adr":    {"num": "initial_adr_cnt",    "denom": "case_count"},
    "persistent_adr": {"num": "persistent_adr_cnt",  "denom": "case_count"},
    "p2p_overturn":   {"num": "p2p_ovrtn_cnt",      "denom": "initial_adr_cnt"},
    "appeal_overturn": {"num": "appeal_ovrtn_cnt",   "denom": "initial_adr_cnt"},
    "mcr_overturn":   {"num": "mcr_ovrtn_cnt",      "denom": "initial_adr_cnt"},
    "md_review":      {"num": "md_reviewed_cnt",     "denom": "case_count"},
}

k_map = {}
for feat, cfg in rate_configs.items():
    k_map[feat] = mom_optimal_k(df_all[cfg["num"]], df_all[cfg["denom"]])

df_26_elig = df_2026.query("case_count >= 30 and initial_adr_cnt >= 10")
k_map["member_appeal"] = mom_optimal_k(df_26_elig["member_appeal_cnt"], df_26_elig["initial_adr_cnt"])

print("k_map:")
for k, v in k_map.items():
    print(f"  {k:<20}: {v:.1f}")

k_map:
  initial_adr         : 20.4
  persistent_adr      : 27.8
  p2p_overturn        : 8.9
  appeal_overturn     : 22.6
  mcr_overturn        : 54.6
  md_review           : 8.7
  member_appeal       : 4.0


In [6]:
iso_kpi_9 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_adj_rate", "persistent_adr_adj_rate", "p2p_overturn_adj_rate",
    "appeal_overturn_adj_rate", "mcr_overturn_adj_rate", "md_review_adj_rate",
    "member_appeal_adj_rate"
]

iso_kpi_8 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_adj_rate", "persistent_adr_adj_rate", "p2p_overturn_adj_rate",
    "appeal_overturn_adj_rate", "mcr_overturn_adj_rate", "md_review_adj_rate",
]

def run_if_and_flag(X, percentile=2.5, random_state=42):
    model = IsolationForest(contamination="auto", n_estimators=300, random_state=random_state)
    model.fit(X)
    scores = model.decision_function(X)
    threshold = np.percentile(scores, percentile)
    return scores < threshold, scores, threshold

In [7]:
# === BASELINE (v4 current): 9 features, member_appeal imputed for 2025 ===
df_iso_25_base = compute_shrinkage(df_2025.copy(), "2025", k_map, features_8=False)
df_iso_26_base = compute_shrinkage(df_2026.copy(), "2026", k_map, features_8=False)

gr_ma_26 = df_iso_26_base["member_appeal_cnt"].sum() / df_iso_26_base["initial_adr_cnt"].sum()
df_iso_25_base["member_appeal_adj_rate"] = gr_ma_26

df_pooled_base = pd.concat([df_iso_25_base, df_iso_26_base], ignore_index=True)
df_pooled_base[iso_kpi_9] = df_pooled_base[iso_kpi_9].replace([np.inf, -np.inf], np.nan).fillna(0)

X_base = df_pooled_base[iso_kpi_9].values
flags_base, _, _ = run_if_and_flag(X_base)
df_pooled_base["flag_baseline"] = flags_base

flagged_25_base = set(
    df_pooled_base.loc[(df_pooled_base["year"] == "2025") & df_pooled_base["flag_baseline"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)
flagged_26_base = set(
    df_pooled_base.loc[(df_pooled_base["year"] == "2026") & df_pooled_base["flag_baseline"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)

print(f"BASELINE (9 feat, imputed): 2025 flagged={len(flagged_25_base)}, 2026 flagged={len(flagged_26_base)}")

BASELINE (9 feat, imputed): 2025 flagged=33, 2026 flagged=18


In [8]:
# === OPTION 1: 8 features, no member_appeal at all ===
df_iso_25_opt1 = compute_shrinkage(df_2025.copy(), "2025", k_map, features_8=True)
df_iso_26_opt1 = compute_shrinkage(df_2026.copy(), "2026", k_map, features_8=True)

df_pooled_opt1 = pd.concat([df_iso_25_opt1, df_iso_26_opt1], ignore_index=True)
df_pooled_opt1[iso_kpi_8] = df_pooled_opt1[iso_kpi_8].replace([np.inf, -np.inf], np.nan).fillna(0)

X_opt1 = df_pooled_opt1[iso_kpi_8].values
flags_opt1, _, _ = run_if_and_flag(X_opt1)
df_pooled_opt1["flag_opt1"] = flags_opt1

flagged_25_opt1 = set(
    df_pooled_opt1.loc[(df_pooled_opt1["year"] == "2025") & df_pooled_opt1["flag_opt1"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)
flagged_26_opt1 = set(
    df_pooled_opt1.loc[(df_pooled_opt1["year"] == "2026") & df_pooled_opt1["flag_opt1"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)

print(f"OPTION 1 (8 feat, no member_appeal): 2025 flagged={len(flagged_25_opt1)}, 2026 flagged={len(flagged_26_opt1)}")

OPTION 1 (8 feat, no member_appeal): 2025 flagged=45, 2026 flagged=6


In [9]:
# === OPTION 2: Same as option 1 for flagging, but add post-hoc member_appeal context ===
# Flagging is identical to option 1 (8 features)
flagged_25_opt2 = flagged_25_opt1.copy()
flagged_26_opt2 = flagged_26_opt1.copy()

# Post-hoc: for 2026 flagged hospitals, report member_appeal rate as context
df_iso_26_full = compute_shrinkage(df_2026.copy(), "2026", k_map, features_8=False)
flagged_26_posthoc = df_pooled_opt1.loc[
    (df_pooled_opt1["year"] == "2026") & df_pooled_opt1["flag_opt1"],
    ["hospital_group", "fin_market"]
].merge(
    df_iso_26_full[["hospital_group", "fin_market", "member_appeal_adj_rate"]],
    on=["hospital_group", "fin_market"],
    how="left"
)

ma_median = df_iso_26_full["member_appeal_adj_rate"].median()
flagged_26_posthoc["ma_vs_median"] = (flagged_26_posthoc["member_appeal_adj_rate"] - ma_median) / ma_median

print(f"OPTION 2 (8 feat + post-hoc): same flags as Option 1")
print(f"\n2026 flagged hospitals — member_appeal context:")
print(flagged_26_posthoc.sort_values("ma_vs_median", ascending=False).to_string(index=False))

OPTION 2 (8 feat + post-hoc): same flags as Option 1

2026 flagged hospitals — member_appeal context:
                hospital_group fin_market  member_appeal_adj_rate  ma_vs_median
          Atrium Health System         NC                0.670268     73.289397
  WESTCHESTER GENERAL HOSPITAL         FL                0.153217     15.981809
      Stanford Health Hospital       CA-N                0.064629      6.163195
ADVOCATE HEALTH & HOSPITALS IL         IL                0.064629      6.163195
            Englewood Hospital         NJ                0.024039      1.664328
              Cape Fear Valley         NC                0.004108     -0.544691


In [10]:
# === OPTION 3: Separate models per year ===
# 2025: 8 features
df_iso_25_opt3 = compute_shrinkage(df_2025.copy(), "2025", k_map, features_8=True)
df_iso_25_opt3[iso_kpi_8] = df_iso_25_opt3[iso_kpi_8].replace([np.inf, -np.inf], np.nan).fillna(0)
X_25_opt3 = df_iso_25_opt3[iso_kpi_8].values
flags_25_opt3, _, _ = run_if_and_flag(X_25_opt3)
df_iso_25_opt3["flag_opt3"] = flags_25_opt3

flagged_25_opt3 = set(
    df_iso_25_opt3.loc[df_iso_25_opt3["flag_opt3"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)

# 2026: 9 features (real member_appeal)
df_iso_26_opt3 = compute_shrinkage(df_2026.copy(), "2026", k_map, features_8=False)
df_iso_26_opt3[iso_kpi_9] = df_iso_26_opt3[iso_kpi_9].replace([np.inf, -np.inf], np.nan).fillna(0)
X_26_opt3 = df_iso_26_opt3[iso_kpi_9].values
flags_26_opt3, _, _ = run_if_and_flag(X_26_opt3)
df_iso_26_opt3["flag_opt3"] = flags_26_opt3

flagged_26_opt3 = set(
    df_iso_26_opt3.loc[df_iso_26_opt3["flag_opt3"],
                       ["hospital_group", "fin_market"]].apply(tuple, axis=1)
)

print(f"OPTION 3 (separate models): 2025 flagged={len(flagged_25_opt3)} (8feat solo), 2026 flagged={len(flagged_26_opt3)} (9feat solo)")

OPTION 3 (separate models): 2025 flagged=31 (8feat solo), 2026 flagged=20 (9feat solo)


In [11]:
def jaccard(a, b):
    if len(a) == 0 and len(b) == 0:
        return 1.0
    return len(a & b) / len(a | b)

print("=" * 70)
print("COMPARISON: 2025 Flagged Sets")
print("=" * 70)
print(f"{'Option':<35} {'Flagged':<10} {'Jaccard vs Base':<18} {'Gained':<10} {'Lost':<10}")
print("-" * 70)

for label, flagged_set in [
    ("Opt1: 8feat pooled", flagged_25_opt1),
    ("Opt2: 8feat pooled + post-hoc", flagged_25_opt2),
    ("Opt3: 8feat 2025-only model", flagged_25_opt3),
]:
    j = jaccard(flagged_25_base, flagged_set)
    gained = flagged_set - flagged_25_base
    lost = flagged_25_base - flagged_set
    print(f"{label:<35} {len(flagged_set):<10} {j:<18.3f} {len(gained):<10} {len(lost):<10}")

print()
print("=" * 70)
print("COMPARISON: 2026 Flagged Sets")
print("=" * 70)
print(f"{'Option':<35} {'Flagged':<10} {'Jaccard vs Base':<18} {'Gained':<10} {'Lost':<10}")
print("-" * 70)

for label, flagged_set in [
    ("Opt1: 8feat pooled", flagged_26_opt1),
    ("Opt2: 8feat pooled + post-hoc", flagged_26_opt2),
    ("Opt3: 9feat 2026-only model", flagged_26_opt3),
]:
    j = jaccard(flagged_26_base, flagged_set)
    gained = flagged_set - flagged_26_base
    lost = flagged_26_base - flagged_set
    print(f"{label:<35} {len(flagged_set):<10} {j:<18.3f} {len(gained):<10} {len(lost):<10}")

COMPARISON: 2025 Flagged Sets
Option                              Flagged    Jaccard vs Base    Gained     Lost      
----------------------------------------------------------------------
Opt1: 8feat pooled                  45         0.696              13         1         
Opt2: 8feat pooled + post-hoc       45         0.696              13         1         
Opt3: 8feat 2025-only model         31         0.778              3          5         

COMPARISON: 2026 Flagged Sets
Option                              Flagged    Jaccard vs Base    Gained     Lost      
----------------------------------------------------------------------
Opt1: 8feat pooled                  6          0.333              0          12        
Opt2: 8feat pooled + post-hoc       6          0.333              0          12        
Opt3: 9feat 2026-only model         20         0.727              4          2         


In [12]:
print("\n" + "=" * 70)
print("DETAIL: Hospitals LOST from 2025 baseline when switching to 8-feature pooled")
print("=" * 70)
lost_25 = flagged_25_base - flagged_25_opt1
if lost_25:
    for hosp, mkt in sorted(lost_25):
        print(f"  {hosp} | {mkt}")
else:
    print("  (none)")

print("\n" + "=" * 70)
print("DETAIL: Hospitals GAINED in 2025 when switching to 8-feature pooled")
print("=" * 70)
gained_25 = flagged_25_opt1 - flagged_25_base
if gained_25:
    for hosp, mkt in sorted(gained_25):
        print(f"  {hosp} | {mkt}")
else:
    print("  (none)")

print("\n" + "=" * 70)
print("DETAIL: Hospitals LOST from 2025 baseline with separate 2025 model (Opt3)")
print("=" * 70)
lost_25_opt3 = flagged_25_base - flagged_25_opt3
if lost_25_opt3:
    for hosp, mkt in sorted(lost_25_opt3):
        print(f"  {hosp} | {mkt}")
else:
    print("  (none)")

print("\n" + "=" * 70)
print("DETAIL: Hospitals GAINED in 2025 with separate 2025 model (Opt3)")
print("=" * 70)
gained_25_opt3 = flagged_25_opt3 - flagged_25_base
if gained_25_opt3:
    for hosp, mkt in sorted(gained_25_opt3):
        print(f"  {hosp} | {mkt}")
else:
    print("  (none)")


DETAIL: Hospitals LOST from 2025 baseline when switching to 8-feature pooled
  McLaren Physician Partners | MI

DETAIL: Hospitals GAINED in 2025 when switching to 8-feature pooled
  ATLANTICARE REG MED CTR | PA
  Adventist Health System Florida | FL
  HCA INC | FL
  HCA INC | TX
  Hoag Memorial Hospital | CA-S
  NJMHMC | NJ
  New York Presbyterian | NY
  PHC-FT MOHAVE | AZ
  RUSSELL COUNTY COMMUNITY HOSPITAL | GA
  SCL Health System | MT
  VAIL CLINIC | CO
  VAUGHAN REGIONAL MEDICAL CTR | AL
  VIRTUA WILLINGBORO HOSPITAL | PA

DETAIL: Hospitals LOST from 2025 baseline with separate 2025 model (Opt3)
  Maury Regional | TN
  McLaren Physician Partners | MI
  Northern Arizona Healthcare | AZ
  Piedmont Health System | GA
  UAB Health System | AL

DETAIL: Hospitals GAINED in 2025 with separate 2025 model (Opt3)
  NJMHMC | NJ
  PHC-FT MOHAVE | AZ
  VIRTUA WILLINGBORO HOSPITAL | PA


In [13]:
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"""\nBaseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: {len(flagged_25_base)} flagged | 2026: {len(flagged_26_base)} flagged

Option 1 (remove member_appeal): 8 feat pooled
  -> 2025: {len(flagged_25_opt1)} flagged | 2026: {len(flagged_26_opt1)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt1):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt1):.3f}

Option 2 (train without + post-hoc): same as Option 1 for flagging
  -> member_appeal reported as supplemental context for 2026 flagged hospitals

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: {len(flagged_25_opt3)} flagged | 2026: {len(flagged_26_opt3)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt3):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt3):.3f}
""")


SUMMARY

Baseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: 33 flagged | 2026: 18 flagged

Option 1 (remove member_appeal): 8 feat pooled
  -> 2025: 45 flagged | 2026: 6 flagged
  -> Jaccard vs baseline: 2025=0.696, 2026=0.333

Option 2 (train without + post-hoc): same as Option 1 for flagging
  -> member_appeal reported as supplemental context for 2026 flagged hospitals

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: 31 flagged | 2026: 20 flagged
  -> Jaccard vs baseline: 2025=0.778, 2026=0.727



In [14]:
def get_hospital_profile(df_pooled, hosp, mkt, features, label):
    row = df_pooled.loc[
        (df_pooled["hospital_group"] == hosp) & (df_pooled["fin_market"] == mkt) &
        (df_pooled["year"] == "2025")
    ]
    if row.empty:
        return None
    row = row.iloc[0]
    medians = df_pooled.loc[df_pooled["year"] == "2025", features].median()
    deviations = []
    for feat in features:
        val = row[feat]
        med = medians[feat]
        pct_dev = (val - med) / med * 100 if med != 0 else 0
        if abs(pct_dev) > 30:
            direction = "HIGH" if pct_dev > 0 else "LOW"
            deviations.append((feat, val, med, pct_dev, direction))
    deviations.sort(key=lambda x: abs(x[3]), reverse=True)
    return deviations

def print_hospital_detail(df_pooled, hospital_set, features, header):
    print("\n" + "=" * 70)
    print(header)
    print("=" * 70)
    if not hospital_set:
        print("  (none)")
        return
    for hosp, mkt in sorted(hospital_set):
        print(f"\n  {hosp} | {mkt}")
        devs = get_hospital_profile(df_pooled, hosp, mkt, features, "")
        if devs:
            print(f"  {'Feature':<30} {'Value':<10} {'Median':<10} {'Dev%':<10} {'Dir':<5}")
            print(f"  {'-'*65}")
            for feat, val, med, pct_dev, direction in devs[:5]:
                print(f"  {feat:<30} {val:<10.4f} {med:<10.4f} {pct_dev:<+10.1f} {direction:<5}")
        else:
            print("    (no features >30% from median)")

# Opt1 vs Baseline
lost_25 = flagged_25_base - flagged_25_opt1
gained_25 = flagged_25_opt1 - flagged_25_base

print_hospital_detail(df_pooled_base, lost_25, iso_kpi_9,
    "LOST from 2025 baseline → 8-feat pooled (Opt1): WHY were these flagged originally?")

print_hospital_detail(df_pooled_opt1, gained_25, iso_kpi_8,
    "GAINED in 2025 with 8-feat pooled (Opt1): WHY are these now flagged?")

# Opt3 vs Baseline
lost_25_opt3 = flagged_25_base - flagged_25_opt3
gained_25_opt3 = flagged_25_opt3 - flagged_25_base

print_hospital_detail(df_pooled_base, lost_25_opt3, iso_kpi_9,
    "LOST from 2025 baseline → separate 2025 model (Opt3): WHY were these flagged originally?")

print_hospital_detail(df_iso_25_opt3, gained_25_opt3, iso_kpi_8,
    "GAINED in 2025 with separate model (Opt3): WHY are these now flagged?")


LOST from 2025 baseline → 8-feat pooled (Opt1): WHY were these flagged originally?

  McLaren Physician Partners | MI
  Feature                        Value      Median     Dev%       Dir  
  -----------------------------------------------------------------
  mcr_overturn_adj_rate          0.0565     0.0227     +148.4     HIGH 
  md_review_adj_rate             0.1344     0.6974     -80.7      LOW  
  initial_adr_adj_rate           0.0758     0.2858     -73.5      LOW  
  persistent_adr_adj_rate        0.0478     0.1472     -67.5      LOW  
  appeal_overturn_adj_rate       0.0027     0.0046     -41.9      LOW  

GAINED in 2025 with 8-feat pooled (Opt1): WHY are these now flagged?

  ATLANTICARE REG MED CTR | PA
  Feature                        Value      Median     Dev%       Dir  
  -----------------------------------------------------------------
  appeal_overturn_adj_rate       0.0361     0.0046     +688.2     HIGH 
  p2p_overturn_adj_rate          0.5466     0.3692     +48.1      H

In [15]:
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"""\nBaseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: {len(flagged_25_base)} flagged | 2026: {len(flagged_26_base)} flagged

Option 1 (remove member_appeal): 8 feat pooled
  -> 2025: {len(flagged_25_opt1)} flagged | 2026: {len(flagged_26_opt1)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt1):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt1):.3f}

Option 2 (train without + post-hoc): same as Option 1 for flagging
  -> member_appeal reported as supplemental context for 2026 flagged hospitals

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: {len(flagged_25_opt3)} flagged | 2026: {len(flagged_26_opt3)} flagged
  -> Jaccard vs baseline: 2025={jaccard(flagged_25_base, flagged_25_opt3):.3f}, 2026={jaccard(flagged_26_base, flagged_26_opt3):.3f}
""")


SUMMARY

Baseline (v4 current): 9 feat, member_appeal imputed for 2025
  -> 2025: 33 flagged | 2026: 18 flagged

Option 1 (remove member_appeal): 8 feat pooled
  -> 2025: 45 flagged | 2026: 6 flagged
  -> Jaccard vs baseline: 2025=0.696, 2026=0.333

Option 2 (train without + post-hoc): same as Option 1 for flagging
  -> member_appeal reported as supplemental context for 2026 flagged hospitals

Option 3 (separate models): 2025=8feat solo, 2026=9feat solo
  -> 2025: 31 flagged | 2026: 20 flagged
  -> Jaccard vs baseline: 2025=0.778, 2026=0.727

